# HTR Experiments Tracking & Analysis

This notebook helps you track, monitor, and analyze all your HTR experiments.

## 1. Setup

In [3]:
import os
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

# Set paths
LOGS_DIR = Path('../experiments_execution/logs')
SAVED_MODELS_DIR = Path('../saved_models/experiments')

print(f"Logs directory: {LOGS_DIR}")
print(f"Saved models directory: {SAVED_MODELS_DIR}")

Logs directory: ../experiments_execution/logs
Saved models directory: ../saved_models/experiments


## 2. Check Running Jobs

In [6]:
# Check SLURM job status
!squeue -u $USER --format="%.18i %.20j %.8T %.10M %.6D %R"

             JOBID                 NAME    STATE       TIME  NODES NODELIST(REASON)
           1514006           trocr_base  PENDING       0:00      1 (Priority)
           1514005           tv_vit_b16  PENDING       0:00      1 (Priority)
           1514004       vit_rgts_16reg  PENDING       0:00      1 (Priority)
           1514003        vit_rgts_8reg  PENDING       0:00      1 (Priority)
           1514002        vit_rgts_4reg  PENDING       0:00      1 (Priority)
           1514001        vit_rgts_2reg  PENDING       0:00      1 (Priority)
           1514000        vit_rgts_0reg  PENDING       0:00      1 (Priority)
           1513999     baseline_cnn_rnn  PENDING       0:00      1 (Priority)


## 3. Monitor Logs in Real-Time

In [5]:
def get_latest_log(log_type='output', experiment_category='vit_rgts_registers'):
    """
    Get the latest log file from a category.
    
    Args:
        log_type: 'output' or 'error'
        experiment_category: 'baseline', 'vit_rgts_registers', or 'pretrained'
    """
    pattern = LOGS_DIR / experiment_category / f"{log_type}_*.log"
    log_files = glob.glob(str(pattern))
    
    if not log_files:
        print(f"No {log_type} logs found in {experiment_category}")
        return None
    
    latest_log = max(log_files, key=os.path.getmtime)
    return latest_log

# Example: View latest output log
latest_log = get_latest_log('output', 'vit_rgts_registers')
if latest_log:
    print(f"Latest log: {latest_log}")
    print("\n" + "="*80)
    with open(latest_log, 'r') as f:
        print(f.read()[-2000:])  # Last 2000 characters

Latest log: ../experiments_execution/logs/vit_rgts_registers/output_16reg_1513136.log

                                |
+-----------------------------------------------------------------------------------------+
### Finished TaskPrologue

Starting ViT-RGTS with 16 Registers
Started at: Sun Jan 18 10:59:04 PM CET 2026

Starting Experiment: run_6
Experiment Directory: ./saved_models/experiments/run_6

🔄 Auto-selected ViT augmentation for vit_rgts
📊 Training with vit augmentation strategy
Completed at: Sun Jan 18 11:17:32 PM CET 2026
=== JOB_STATISTICS ===
=== current date     : Sun Jan 18 11:17:32 PM CET 2026
= Job-ID             : 1513136 on tinygpu
= Job-Name           : vit_rgts_16reg
= Job-Command        : /home/hpc/iwi5/iwi5369h/HTR-Pipeline/experiments_execution/slurm_scripts/06_vit_rgts_16reg.slurm
= Initial workdir    : /home/hpc/iwi5/iwi5369h/HTR-Pipeline/experiments_execution/slurm_scripts
= Queue/Partition    : rtx3080
= Slurm account      : iwi5 with QOS=normal
= Requested r

## 4. List All Experiments

In [4]:
def list_all_logs():
    """List all experiment logs organized by category."""
    categories = ['baseline', 'vit_rgts_registers', 'pretrained']
    
    all_logs = {}
    
    for category in categories:
        cat_dir = LOGS_DIR / category
        if cat_dir.exists():
            output_logs = glob.glob(str(cat_dir / "output_*.log"))
            error_logs = glob.glob(str(cat_dir / "error_*.log"))
            
            all_logs[category] = {
                'output': sorted(output_logs, key=os.path.getmtime, reverse=True),
                'error': sorted(error_logs, key=os.path.getmtime, reverse=True)
            }
    
    return all_logs

# Display all logs
logs = list_all_logs()

for category, log_types in logs.items():
    print(f"\n{'='*80}")
    print(f"Category: {category.upper()}")
    print(f"{'='*80}")
    
    print(f"\nOutput logs ({len(log_types['output'])}):") 
    for log in log_types['output'][:5]:  # Show latest 5
        mod_time = datetime.fromtimestamp(os.path.getmtime(log))
        print(f"  {os.path.basename(log):40} - {mod_time}")
    
    print(f"\nError logs ({len(log_types['error'])}):") 
    for log in log_types['error'][:5]:  # Show latest 5
        mod_time = datetime.fromtimestamp(os.path.getmtime(log))
        print(f"  {os.path.basename(log):40} - {mod_time}")


Category: BASELINE

Output logs (0):

Error logs (0):

Category: VIT_RGTS_REGISTERS

Output logs (0):

Error logs (0):

Category: PRETRAINED

Output logs (0):

Error logs (0):


## 5. Analyze Experiment Results

In [ ]:
def load_experiment_results():
    """Load results from all completed experiments."""
    results = []
    
    for run_dir in sorted(SAVED_MODELS_DIR.glob('run_*')):
        config_file = run_dir / 'config.json'
        results_file = run_dir / 'evaluation_details.csv'
        
        if config_file.exists():
            with open(config_file, 'r') as f:
                config = json.load(f)
            
            # Extract key info
            result = {
                'run': run_dir.name,
                'architecture': config.get('arch', {}).get('name', 'unknown'),
                'num_registers': config.get('arch', {}).get('num_registers', None),
            }
            
            # Load evaluation results if available
            if results_file.exists():
                eval_df = pd.read_csv(results_file)
                if len(eval_df) > 0:
                    result['cer'] = eval_df['cer'].mean()
                    result['wer'] = eval_df['wer'].mean()
            
            results.append(result)
    
    return pd.DataFrame(results)

# Load and display results
results_df = load_experiment_results()

if len(results_df) > 0:
    print("Experiment Results Summary:")
    print("="*80)
    print(results_df.to_string(index=False))
else:
    print("No completed experiments found yet.")

## 6. Visualize Register Token Impact

In [ ]:
# Filter ViT-RGTS results
if len(results_df) > 0 and 'num_registers' in results_df.columns:
    vit_rgts_results = results_df[
        (results_df['architecture'] == 'vit_rgts') & 
        (results_df['cer'].notna())
    ].copy()
    
    if len(vit_rgts_results) > 0:
        vit_rgts_results = vit_rgts_results.sort_values('num_registers')
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # CER vs Registers
        ax1.plot(vit_rgts_results['num_registers'], vit_rgts_results['cer'], 
                marker='o', linewidth=2, markersize=8)
        ax1.set_xlabel('Number of Register Tokens', fontsize=12)
        ax1.set_ylabel('Character Error Rate (%)', fontsize=12)
        ax1.set_title('CER vs Register Tokens', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # WER vs Registers
        ax2.plot(vit_rgts_results['num_registers'], vit_rgts_results['wer'], 
                marker='s', linewidth=2, markersize=8, color='orange')
        ax2.set_xlabel('Number of Register Tokens', fontsize=12)
        ax2.set_ylabel('Word Error Rate (%)', fontsize=12)
        ax2.set_title('WER vs Register Tokens', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('register_token_impact.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("\nBest Configuration:")
        best_idx = vit_rgts_results['cer'].idxmin()
        best_result = vit_rgts_results.loc[best_idx]
        print(f"  Registers: {best_result['num_registers']}")
        print(f"  CER: {best_result['cer']:.2f}%")
        print(f"  WER: {best_result['wer']:.2f}%")
    else:
        print("No ViT-RGTS results with evaluation metrics found yet.")
else:
    print("No results available yet for visualization.")

## 7. Compare All Architectures

In [ ]:
if len(results_df) > 0 and 'cer' in results_df.columns:
    valid_results = results_df[results_df['cer'].notna()].copy()
    
    if len(valid_results) > 0:
        # Create model labels
        valid_results['model_label'] = valid_results.apply(
            lambda x: f"{x['architecture']} ({x['num_registers']}r)" 
                     if pd.notna(x['num_registers']) 
                     else x['architecture'],
            axis=1
        )
        
        valid_results = valid_results.sort_values('cer')
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        x_pos = range(len(valid_results))
        ax.bar(x_pos, valid_results['cer'], alpha=0.7, label='CER')
        ax.bar(x_pos, valid_results['wer'], alpha=0.7, label='WER')
        
        ax.set_xlabel('Model', fontsize=12)
        ax.set_ylabel('Error Rate (%)', fontsize=12)
        ax.set_title('Architecture Comparison', fontsize=14, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(valid_results['model_label'], rotation=45, ha='right')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig('architecture_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print("No valid results with CER/WER metrics found yet.")
else:
    print("No results available yet for comparison.")

## 8. Quick Status Check

In [ ]:
def experiment_status():
    """Quick status overview of all experiments."""
    print("="*80)
    print("EXPERIMENT STATUS DASHBOARD")
    print("="*80)
    
    # Count log files
    logs = list_all_logs()
    total_outputs = sum(len(v['output']) for v in logs.values())
    total_errors = sum(len(v['error']) for v in logs.values())
    
    print(f"\nTotal Output Logs: {total_outputs}")
    print(f"Total Error Logs: {total_errors}")
    
    # Count completed experiments
    completed_runs = len(list(SAVED_MODELS_DIR.glob('run_*/model.pt')))
    print(f"\nCompleted Experiments (with model.pt): {completed_runs}")
    
    # Show running jobs
    print("\nCurrently Running Jobs:")
    os.system("squeue -u $USER --format='  %.18i %.20j %.8T %.10M'")
    
    # Latest activity
    all_logs_flat = []
    for cat_logs in logs.values():
        all_logs_flat.extend(cat_logs['output'])
    
    if all_logs_flat:
        latest = max(all_logs_flat, key=os.path.getmtime)
        mod_time = datetime.fromtimestamp(os.path.getmtime(latest))
        print(f"\nLatest Activity: {os.path.basename(latest)}")
        print(f"Time: {mod_time}")
    
    print("\n" + "="*80)

experiment_status()

## 9. Utilities

In [ ]:
# Cancel all your jobs
# !scancel -u $USER

# View specific log file
# log_file = 'experiments_execution/logs/vit_rgts_registers/output_4reg_12345.log'
# !cat {log_file}

# Monitor real-time (run in terminal, not notebook)
# !tail -f experiments_execution/logs/*/output_*.log